# Subcluster neurons
We sublcuster neurons, starting either with the neuron labels from scANVI or from seurat label transfer. We move forward with the option using Seurat transferred labels.

**Pinned Environment:** [`envs/sc-scvi.yaml`](../../envs/sc-scvi.yaml)  

In [ ]:
from pathlib import Path
import os
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import sparse
import warnings
import session_info
import sys

In [ ]:
plt.rcParams['figure.figsize'] = (3, 3)
plt.rcParams['figure.dpi'] = 150

### Setup

In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[1]))

from config.paths import BASE_DIR

input_dir = BASE_DIR / "data/h5ad/export_04/04a_neurons"
adata_path = input_dir / "scanvi_neuron_adata.h5ad"
bdata_path = input_dir / "seurat_neuron_adata.h5ad"

output_dir = BASE_DIR / "data/h5ad/export_04/04b_reclustered"
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(adata_path)
bdata = sc.read_h5ad(bdata_path)

In [ ]:
print(adata)
print("-"*40)
print(bdata)

# Neighbors graph

In [ ]:
sc.pp.neighbors(adata, use_rep = 'X_scVI_scanvi_neuron', random_state = 0)
sc.pp.neighbors(bdata, use_rep = 'X_scVI_seurat_neuron', random_state = 0)

In [ ]:
sc.tl.umap(adata, random_state = 0)
sc.tl.umap(bdata, random_state = 0)

In [ ]:
sc.tl.leiden(adata, key_added='neuron_recluster_scanvi', resolution=0.25)
sc.tl.leiden(bdata, key_added='neuron_recluster_seurat', resolution=0.25)

In [ ]:
def assign_cell_type_colors(adata, key="cell_type"):
    """
    Assigns a tab10 color palette to a categorical obs field.
    If there are more than 10 categories, the palette cycles.
    """
    # Ensure categorical
    adata.obs[key] = adata.obs[key].astype("category")
    
    cats = adata.obs[key].cat.categories
    n = len(cats)

    # tab10 gives 10 colors; using modulo lets us cycle if n > 1=20
    base_palette = sns.color_palette("tab20", 20)

    palette = [mcolors.to_hex(base_palette[i % 20]) for i in range(n)]

    # assign colors in category order
    adata.uns[f"{key}_colors"] = palette

In [ ]:
assign_cell_type_colors(adata,  key="scanvi_labels")
assign_cell_type_colors(adata,  key="seurat_labels")
assign_cell_type_colors(bdata,  key="scanvi_labels")
assign_cell_type_colors(bdata,  key="seurat_labels")

In [ ]:
sc.pl.umap(adata, color = 'scanvi_labels', frameon = False, title = 'scANVI labels')
sc.pl.umap(bdata, color = 'seurat_labels', frameon = False, title = 'Seurat labels')
sc.pl.umap(adata, color = 'neuron_recluster_scanvi', frameon = False, title = 'Neuron cluster, scANVI')
sc.pl.umap(bdata, color = 'neuron_recluster_seurat', frameon = False, title = 'Neuron cluster, Seurat')

# Export

In [ ]:
adata_path = os.path.join(output_dir, 'neurons-scanvi.h5ad')
bdata_path = os.path.join(output_dir, 'neurons-seurat.h5ad')

adata.write_h5ad(adata_path, compression='gzip')
bdata.write_h5ad(bdata_path, compression='gzip')

print(adata_path)
print(bdata_path)